# Day 19 — Cross-validation and robust validation practices

Run each cell with **Shift + Enter**, from the top. We use `student_data_150.csv`: 8 real students and 142 **synthetic** ones.

## 1. Cross-validation fundamentals

In [ ]:
# Get the student dataset: Colab starts with an empty folder. If the download fails, upload student_data_150.csv
# with the Files panel (the folder icon on the left) and run this cell again
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-019/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# The 23 test students are set aside first, as on Days 17 and 18; everything today happens inside the other 127
students = pd.read_csv("student_data_150.csv")
features = ["StudyHours", "Attendance", "PreviousScore"]
pool, test = train_test_split(students, test_size=0.15, random_state=42, stratify=students["Passed"])
X, y = pool[features], pool["Passed"].astype(int)
model = Pipeline([("scaler", StandardScaler()), ("classifier", LogisticRegression(max_iter=1000))])

# One split, twenty times: the same model, the same 127 students, a different random 80/20 split each time
accuracies = []
for seed in range(20):
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    accuracies.append(model.fit(X_tr, y_tr).score(X_val, y_val))
print("Validation students per split:", len(X_val))
print("Accuracy on 20 different splits:")
print(np.round(accuracies[:10], 2))
print(np.round(accuracies[10:], 2))
print(f"Lowest {min(accuracies):.3f} | highest {max(accuracies):.3f} | mean {np.mean(accuracies):.3f}")

Validation students per split: 26
Accuracy on 20 different splits:
[0.85 0.92 0.73 0.92 0.73 0.73 0.69 0.58 0.73 0.81]
[0.85 0.65 0.81 0.69 0.73 0.77 0.77 0.65 0.81 0.77]
Lowest 0.577 | highest 0.923 | mean 0.760


In [ ]:
# The lesson's five fold scores: mean and spread
fold_scores = np.array([0.88, 0.91, 0.86, 0.90, 0.85])
print("Mean:", fold_scores.mean().round(3))
print("Standard deviation (np.std, as scikit-learn reports):", fold_scores.std().round(4))
print("Sample standard deviation (ddof=1):                 ", fold_scores.std(ddof=1).round(4))

Mean: 0.88
Standard deviation (np.std, as scikit-learn reports): 0.0228
Sample standard deviation (ddof=1):                  0.0255


In [ ]:
from sklearn.model_selection import KFold, cross_val_score

# 5-fold cross-validation: every student is validated once and trained on four times
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=kfold, scoring="accuracy")
print("Fold scores:", scores.round(3))
print(f"Mean accuracy {scores.mean():.3f} | standard deviation {scores.std():.3f}")
print("Validation fold sizes:", [len(v) for _, v in kfold.split(X)])

Fold scores: [0.846 0.692 0.76  0.64  0.92 ]
Mean accuracy 0.772 | standard deviation 0.101
Validation fold sizes: [26, 26, 25, 25, 25]


In [ ]:
from sklearn.model_selection import StratifiedKFold

# 37 of the 127 students fail (29%). How many failing students land in each validation fold?
print(f"Failing overall: {(y == 0).sum()} of {len(y)} ({(y == 0).mean():.0%})")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, splitter in [("KFold", kfold), ("Stratified", skf)]:
    fails = [int((y.iloc[v] == 0).sum()) for _, v in splitter.split(X, y)]
    print(f"{name:<10} failing per fold {fails}")

Failing overall: 37 of 127 (29%)
KFold      failing per fold [5, 7, 10, 8, 7]
Stratified failing per fold [8, 8, 7, 7, 7]


In [ ]:
from sklearn.metrics import make_scorer, f1_score

# F1 for fail (the class we want to catch) per fold, with plain and stratified 5-fold
fail_f1 = make_scorer(f1_score, pos_label=0)
for name, splitter in [("KFold", kfold), ("Stratified", skf)]:
    s = cross_val_score(model, X, y, cv=splitter, scoring=fail_f1)
    print(f"{name:<10} {s.round(2)} mean {s.mean():.3f} std {s.std():.3f}")

KFold      [0.6  0.5  0.62 0.31 0.86] mean 0.578 std 0.179
Stratified [0.59 0.67 0.36 0.73 0.75] mean 0.619 std 0.140


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Preprocessing leakage: a scaler fitted on all 127 students before cross-validation has seen every validation fold
knn = KNeighborsClassifier()
leaky = cross_val_score(knn, StandardScaler().fit_transform(X), y, cv=skf).mean()
inside = cross_val_score(Pipeline([("scaler", StandardScaler()), ("classifier", knn)]), X, y, cv=skf).mean()
print(f"Scaler fitted before CV:  {leaky:.3f}")
print(f"Scaler inside the pipeline: {inside:.3f}")

Scaler fitted before CV:  0.771
Scaler inside the pipeline: 0.795
